## Accuracy

In [13]:
import numpy as np
import glob
import matplotlib.pyplot as plt
from sklearn.metrics import normalized_mutual_info_score
from sklearn.metrics import adjusted_rand_score
import pandas as pd
import rdata
from pathlib import Path


folder_pen = Path("ouputfinal/penalized")

rdata_files_pen = sorted(folder_pen.glob("*.Rdata"))
all_data_pen_censoradministrative = []
all_data_pen_censornormal = []


# ------------------------
# LOAD RDATA
# ------------------------

for file in rdata_files_pen:
    if "censoradministrative" in str(file):
        parsed = rdata.parser.parse_file(file)
        converted = rdata.conversion.convert(parsed)
        all_data_pen_censoradministrative.append(converted)

    elif "censornormal" in str(file):
        parsed = rdata.parser.parse_file(file)
        converted = rdata.conversion.convert(parsed)
        all_data_pen_censornormal.append(converted)



# ------------------------
# SIMPLE RELABEL
# ------------------------

def simple_relabel(true_labels, pred_labels):

    true_labels = pd.Series(true_labels)
    pred_labels = pd.Series(pred_labels)

    mapping = {}
    available_true = set(true_labels.unique())

    for p in pred_labels.unique():

        overlaps = {t: ((true_labels==t) & (pred_labels==p)).sum()
                     for t in available_true}

        best = max(overlaps, key=overlaps.get)
        mapping[p] = best
        available_true.remove(best)

    return pred_labels.map(mapping)



# ------------------------
# COLLECT METRICS BY (censoring, k, gamma)
# ------------------------

def process_list(dataset_list, censor_name):

    results = {}   # (k, gamma) -> list of accuracy, ari

    for d in dataset_list:

        if d["n_components"] != 3:
            continue

        k = int(np.asarray(d["k"]).item())
        gamma = float(np.asarray(d["gammapar"]).item())

        df = pd.DataFrame(d["clusters"])
        true_labels = df.iloc[:, 0]
        pred_labels = df.iloc[:, 1]

        aligned = simple_relabel(true_labels, pred_labels)

        accuracy = (true_labels == aligned).mean()
        ari = adjusted_rand_score(true_labels, aligned)

        key = (censor_name, k, gamma)
        if key not in results:
            results[key] = {"acc": [], "ari": []}

        results[key]["acc"].append(accuracy)
        results[key]["ari"].append(ari)

    return results



admin_results = process_list(all_data_pen_censoradministrative, "Administrative")
norm_results  = process_list(all_data_pen_censornormal, "Normal")



# ------------------------
# BUILD TABLE-READY SUMMARY
# ------------------------

rows = []

def add_rows(result_dict):
    for (censor, k, gamma), vals in result_dict.items():
        acc = np.array(vals["acc"])
        ari = np.array(vals["ari"])

        rows.append({
            "Censoring": censor,
            "k": k,
            "gamma": gamma,
            "Acc_mean": np.round(acc.mean(),3),
            "Acc_median": np.round(np.median(acc),3),
            "Acc_sd": np.round(acc.std(),3),
            "ARI_mean": np.round(ari.mean(),3),
            "ARI_median": np.round(np.median(ari),3),
            "ARI_sd": np.round(ari.std(),3)
        })

add_rows(admin_results)
add_rows(norm_results)

summary_df = pd.DataFrame(rows).sort_values(["Censoring", "k", "gamma"])

print(summary_df)


/Users/alessandragni/Library/Python/3.9/lib/python/site-packages/rdata/parser/_parser.py:1217: UserWarning: Unknown file type: assumed RDS
  warnings.warn("Unknown file type: assumed RDS")  # noqa: B028
/Users/alessandragni/Library/Python/3.9/lib/python/site-packages/rdata/parser/_parser.py:1220: UserWarning: Wrong extension .Rdata for file in RDS format
  warnings.warn(f"Wrong extension {extension} for file in RDS format")  # noqa: B028
/Users/alessandragni/Library/Python/3.9/lib/python/site-packages/rdata/conversion/_conversion.py:856: UserWarning: Missing constructor for R class "table". The underlying R object is returned instead.
  warnings.warn(


         Censoring   k     gamma  Acc_mean  Acc_median  Acc_sd  ARI_mean  \
6   Administrative  20  0.000001     0.971       1.000   0.089     0.942   
5   Administrative  20  0.000100     0.971       1.000   0.089     0.942   
0   Administrative  20  0.001000     0.975       1.000   0.081     0.950   
1   Administrative  20  0.010000     0.957       1.000   0.082     0.907   
2   Administrative  20  0.100000     0.950       1.000   0.083     0.888   
3   Administrative  20  0.200000     0.930       1.000   0.115     0.860   
4   Administrative  20  0.400000     0.892       0.928   0.141     0.788   
13  Administrative  50  0.000001     1.000       1.000   0.001     0.999   
12  Administrative  50  0.000100     1.000       1.000   0.001     0.999   
7   Administrative  50  0.001000     1.000       1.000   0.001     0.999   
8   Administrative  50  0.010000     1.000       1.000   0.001     0.999   
9   Administrative  50  0.100000     0.996       1.000   0.023     0.989   
10  Administ

In [ ]:
import numpy as np
import glob
import matplotlib.pyplot as plt
from sklearn.metrics import normalized_mutual_info_score
from sklearn.metrics import adjusted_rand_score
import pandas as pd
import rdata
from pathlib import Path

# Folder containing your .RData files
folder_pen = Path("ouputfinal/penalized")

# List all .Rdata files
rdata_files_pen = sorted(folder_pen.glob("*.Rdata"))
all_data_pen_censoradministrative = []
all_data_pen_censornormal = []
all_data_pen_censoruniform = []


for file in rdata_files_pen:
    if "censoradministrative" in str(file):
        parsed = rdata.parser.parse_file(file)
        converted = rdata.conversion.convert(parsed)
        all_data_pen_censoradministrative.append(converted)
    elif "censornormal" in str(file):
        parsed = rdata.parser.parse_file(file)
        converted = rdata.conversion.convert(parsed)
        all_data_pen_censornormal.append(converted)
    elif "censoruniform" in str(file):
        parsed = rdata.parser.parse_file(file)
        converted = rdata.conversion.convert(parsed)
        all_data_pen_censoruniform.append(converted)



/Users/alessandragni/Library/Python/3.9/lib/python/site-packages/rdata/parser/_parser.py:1217: UserWarning: Unknown file type: assumed RDS
  warnings.warn("Unknown file type: assumed RDS")  # noqa: B028
/Users/alessandragni/Library/Python/3.9/lib/python/site-packages/rdata/parser/_parser.py:1220: UserWarning: Wrong extension .Rdata for file in RDS format
  warnings.warn(f"Wrong extension {extension} for file in RDS format")  # noqa: B028
/Users/alessandragni/Library/Python/3.9/lib/python/site-packages/rdata/conversion/_conversion.py:856: UserWarning: Missing constructor for R class "table". The underlying R object is returned instead.
  warnings.warn(


In [8]:
def simple_relabel(true_labels, pred_labels):
    """
    Relabel predicted clusters to match true labels based on maximum overlap.
    Works when number of clusters is small (e.g., 3) and you just want names to match.
    """
    true_labels = pd.Series(true_labels)
    pred_labels = pd.Series(pred_labels)
    
    unique_true = true_labels.unique()
    unique_pred = pred_labels.unique()
    
    mapping = {}
    used_true = set()
    
    for p in unique_pred:
        # find the true label with max overlap
        overlaps = {t: ((true_labels==t) & (pred_labels==p)).sum() for t in unique_true if t not in used_true}
        best_true = max(overlaps, key=overlaps.get)
        mapping[p] = best_true
        used_true.add(best_true)
    
    new_preds = pred_labels.map(mapping)
    return new_preds, mapping

In [9]:
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score

gammapars_all = {}
accuracy_all = {}
ari_all = {}
nmi_all = {}

numb = {
    'admin': "(i) Administrative censoring",
    'norm': "(ii) Normal censoring",
    'unif': "(iii) Uniform censoring"
}

# (all_data_pen_censoruniform, "unif")

for j,name in [(all_data_pen_censoradministrative, "admin"), (all_data_pen_censornormal, "norm")]:
    datasets = []
    gammapars = []
    for d in j:
        if d["n_components"] == 3 and d["k"] == 20:
            if d['gammapar'] != 0:
                datasets.append(pd.DataFrame(d["clusters"]))
                gammapars.append(d["gammapar"][0])

        
    accuracy_list = []
    nmi_list = []
    ari_list = []

    for df in datasets:
        # Extract the two cluster membership columns
        true_labels = df.iloc[:, 0]
        pred_labels = df.iloc[:, 1]

        # Align predicted clusters to true clusters
        aligned_preds, mapping = simple_relabel(true_labels, pred_labels)

        # Add an indicator column for exact matches
        df["same_membership"] = (true_labels == aligned_preds).astype(int)


        # Compute metrics
        nmi = normalized_mutual_info_score(true_labels, aligned_preds)
        print(f"NMI: {nmi:.3f}")
        nmi_list.append(nmi)

        ari = adjusted_rand_score(true_labels, aligned_preds)
        print(f"ARI: {ari:.3f}") 
        ari_list.append(ari)   

        accuracy = df["same_membership"].mean()
        print(f"Accuracy: {accuracy:.3f}")   
        accuracy_list.append(accuracy)

        print('\n')

    
    gammapars_all[f"{name}"] = np.array(gammapars)
    accuracy_all[f"{name}"] = np.array(accuracy_list)
    ari_all[f"{name}"] = np.array(ari_list)
    nmi_all[f"{name}"] = np.array(nmi_list)


NMI: 1.000
ARI: 1.000
Accuracy: 1.000


NMI: 1.000
ARI: 1.000
Accuracy: 1.000


NMI: 1.000
ARI: 1.000
Accuracy: 1.000


NMI: 1.000
ARI: 1.000
Accuracy: 1.000


NMI: 0.837
ARI: 0.819
Accuracy: 0.928


NMI: 1.000
ARI: 1.000
Accuracy: 1.000


NMI: 1.000
ARI: 1.000
Accuracy: 1.000


NMI: 1.000
ARI: 1.000
Accuracy: 1.000


NMI: 0.713
ARI: 0.583
Accuracy: 0.790


NMI: 1.000
ARI: 1.000
Accuracy: 1.000


NMI: 0.854
ARI: 0.839
Accuracy: 0.942


NMI: 0.786
ARI: 0.723
Accuracy: 0.890


NMI: 1.000
ARI: 1.000
Accuracy: 1.000


NMI: 1.000
ARI: 1.000
Accuracy: 1.000


NMI: 1.000
ARI: 1.000
Accuracy: 1.000


NMI: 1.000
ARI: 1.000
Accuracy: 1.000


NMI: 1.000
ARI: 1.000
Accuracy: 1.000


NMI: 1.000
ARI: 1.000
Accuracy: 1.000


NMI: 1.000
ARI: 1.000
Accuracy: 1.000


NMI: 1.000
ARI: 1.000
Accuracy: 1.000


NMI: 1.000
ARI: 1.000
Accuracy: 1.000


NMI: 1.000
ARI: 1.000
Accuracy: 1.000


NMI: 1.000
ARI: 1.000
Accuracy: 1.000


NMI: 0.748
ARI: 0.660
Accuracy: 0.844


NMI: 0.781
ARI: 0.722
Accuracy: 0.884




In [4]:
accuracy_all

{'admin': array([1.   , 1.   , 1.   , 1.   , 0.928, 1.   , 1.   , 1.   , 0.79 ,
        1.   , 0.942, 0.89 , 1.   , 1.   , 1.   , 1.   , 1.   , 1.   ,
        1.   , 1.   , 1.   , 1.   , 1.   , 0.844, 0.884, 0.998, 1.   ,
        1.   , 1.   , 1.   , 1.   , 1.   , 0.834, 1.   , 1.   , 1.   ,
        1.   , 1.   , 1.   , 0.892, 1.   , 1.   , 0.886, 0.886, 0.892,
        0.822, 0.71 , 0.536, 0.536, 1.   , 1.   , 1.   , 1.   , 0.764,
        1.   , 1.   , 0.61 , 0.77 , 0.712, 0.858, 0.92 , 0.61 , 0.61 ,
        0.848, 0.692, 0.716, 0.74 , 0.516, 0.848, 0.848, 1.   , 1.   ,
        1.   , 1.   , 1.   , 1.   , 1.   , 1.   , 1.   , 1.   , 1.   ,
        1.   , 1.   , 1.   , 1.   , 0.726, 0.888, 0.892, 0.874, 1.   ,
        1.   , 0.998, 0.896, 0.998, 0.856, 0.998, 0.998, 1.   , 1.   ,
        1.   , 1.   , 1.   , 1.   , 1.   , 1.   , 1.   , 1.   , 0.892,
        0.692, 1.   , 1.   , 1.   , 1.   , 1.   , 1.   , 1.   , 1.   ,
        1.   , 1.   , 1.   , 1.   , 1.   , 0.692, 1.   , 1.   , 1.  

In [5]:
gammapars_all

{'admin': array([1.e-03, 1.e-02, 1.e-01, 2.e-01, 4.e-01, 1.e-04, 1.e-06, 1.e-03,
        1.e-02, 1.e-01, 2.e-01, 4.e-01, 1.e-04, 1.e-06, 1.e-03, 1.e-02,
        1.e-01, 2.e-01, 4.e-01, 1.e-04, 1.e-06, 1.e-03, 1.e-02, 1.e-01,
        2.e-01, 4.e-01, 1.e-04, 1.e-06, 1.e-03, 1.e-02, 1.e-01, 2.e-01,
        4.e-01, 1.e-04, 1.e-06, 1.e-03, 1.e-02, 1.e-01, 2.e-01, 4.e-01,
        1.e-04, 1.e-06, 1.e-03, 1.e-02, 1.e-01, 2.e-01, 4.e-01, 1.e-04,
        1.e-06, 1.e-03, 1.e-02, 1.e-01, 2.e-01, 4.e-01, 1.e-04, 1.e-06,
        1.e-03, 1.e-02, 1.e-01, 2.e-01, 4.e-01, 1.e-04, 1.e-06, 1.e-03,
        1.e-02, 1.e-01, 2.e-01, 4.e-01, 1.e-04, 1.e-06, 1.e-03, 1.e-02,
        1.e-01, 2.e-01, 4.e-01, 1.e-04, 1.e-06, 1.e-03, 1.e-02, 1.e-01,
        2.e-01, 4.e-01, 1.e-04, 1.e-06, 1.e-03, 1.e-02, 1.e-01, 2.e-01,
        4.e-01, 1.e-04, 1.e-06, 1.e-03, 1.e-02, 1.e-01, 2.e-01, 1.e-04,
        1.e-06, 1.e-03, 1.e-02, 1.e-01, 2.e-01, 4.e-01, 1.e-04, 1.e-06,
        1.e-03, 1.e-02, 1.e-01, 2.e-01, 4.e-01, 1.e-04,

In [295]:
# for j in ['admin', 'norm', 'unif']:

#     # Create a DataFrame for grouping
#     df_acc = pd.DataFrame({"Accuracy": accuracy_all[j], "Gamma": gammapars_all[j]})

#     plt.figure(figsize=(6, 5))

#     # Use seaborn boxplot with greyscale palette for consistency
#     sns.boxplot(
#         data=df_acc,
#         x="Gamma",
#         y="Accuracy",
#         palette="Greys"
#     )



#     # Styling
#     plt.title(f"{numb[j]}", fontsize=TITLE_SIZE)
#     plt.xlabel(r"$\gamma$", fontsize=LABEL_SIZE)
#     plt.ylabel("Accuracy", fontsize=LABEL_SIZE)
#     plt.grid(axis='y', linestyle='--', alpha=0.5)

#     plt.xticks(fontsize=TICK_SIZE)
#     plt.yticks(fontsize=TICK_SIZE)

#     plt.tight_layout()

#     # Save the figure (PDF for LaTeX)
#     plt.savefig(f"Plots/fig_Accuracy_{j}.pdf", bbox_inches="tight")
#     plt.close()



In [296]:
# for j in ['admin', 'norm', 'unif']:

#     # Create a DataFrame for grouping
#     df_acc = pd.DataFrame({"ARI": ari_all[j], "Gamma": gammapars_all[j]})

#     plt.figure(figsize=(6, 5))

#     # Use seaborn boxplot with greyscale palette for consistency
#     sns.boxplot(
#         data=df_acc,
#         x="Gamma",
#         y="ARI",
#         palette="Greys"
#     )

#     # Styling
#     plt.title(f"{numb[j]}", fontsize=TITLE_SIZE)
#     plt.xlabel(r"$\gamma$", fontsize=LABEL_SIZE)
#     plt.ylabel("ARI", fontsize=LABEL_SIZE)
#     plt.grid(axis='y', linestyle='--', alpha=0.5)

#     plt.xticks(fontsize=TICK_SIZE)
#     plt.yticks(fontsize=TICK_SIZE)

#     plt.tight_layout()

#     # Save the figure (PDF for LaTeX)
#     plt.savefig(f"Plots/fig_ARI_{j}.pdf", bbox_inches="tight")
#     plt.close()

In [297]:
# for j in ['admin', 'norm', 'unif']:

#     # Create a DataFrame for grouping
#     df_acc = pd.DataFrame({"NMI": nmi_all[j], "Gamma": gammapars_all[j]})

#     plt.figure(figsize=(6, 5))

#     # Use seaborn boxplot with greyscale palette for consistency
#     sns.boxplot(
#         data=df_acc,
#         x="Gamma",
#         y="NMI",
#         palette="Greys"
#     )


#     # Styling
#     plt.title(f"{numb[j]}", fontsize=TITLE_SIZE)
#     plt.xlabel(r"$\gamma$", fontsize=LABEL_SIZE)
#     plt.ylabel("NMI", fontsize=LABEL_SIZE)
#     plt.grid(axis='y', linestyle='--', alpha=0.5)

#     plt.xticks(fontsize=TICK_SIZE)
#     plt.yticks(fontsize=TICK_SIZE)

#     plt.tight_layout()

#     # Save the figure (PDF for LaTeX)
#     plt.savefig(f"Plots/fig_NMI_{j}.pdf", bbox_inches="tight")
#     plt.close()

In [72]:
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

titles_acc = {
    'admin': "(i) Accuracy – Administrative cens.",
    'norm':  "(ii) Accuracy – Normal cens.",
}

titles_ari = {
    'admin': "(iii) ARI – Administrative cens.",
    'norm':  "(iv) ARI – Normal cens.",
}

# -------------------------------
# Create 2×2 figure
# -------------------------------
fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=False)

# store gamma values once
gammas_used = None

# --------------------------------
# First row: ACCURACY
# --------------------------------
for col, j in enumerate(['admin', 'norm']):

    df_acc = pd.DataFrame({
        "Value": accuracy_all[j],
        "Gamma": gammapars_all[j]
    })

    if gammas_used is None:
        gammas_used = np.sort(df_acc["Gamma"].unique())

    sns.boxplot(
        data=df_acc,
        x="Gamma",
        y="Value",
        palette="Greys",
        ax=axes[col]
    )

    ax = axes[col]
    ax.set_title(titles_acc[j], fontsize=TITLE_SIZE-5)
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticks([])
    ax.set_xticklabels([])

# --------------------------------
# Second row: ARI
# --------------------------------
for col, j in enumerate(['admin', 'norm']):

    df_ari = pd.DataFrame({
        "Value": ari_all[j],        # your ARI values here
        "Gamma": gammapars_all[j]
    })

    sns.boxplot(
        data=df_ari,
        x="Gamma",
        y="Value",
        palette="Greys",
        ax=axes[col+2]
    )

    ax = axes[col+2]
    ax.set_title(titles_ari[j], fontsize=TITLE_SIZE-5)
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticks([])
    ax.set_xticklabels([])

# --------------------------------
# Build the global gamma legend
# --------------------------------
colors = sns.color_palette("Greys", len(gammas_used))

labels = []
for g in gammas_used:
    if g == 0:
        labels.append("0")
    elif abs(g) < 0.01:
        e = int(np.floor(np.log10(abs(g))))
        labels.append(rf"$10^{{{e}}}$")
    else:
        labels.append(f"{g:g}")

handles = [
    mpatches.Patch(facecolor=colors[k], label=labels[k], 
                   edgecolor="black",  
                   linewidth=0.5)
    for k in range(len(gammas_used))
]

fig.legend(
    handles,
    labels,
    title=r"$\gamma$",
    loc="lower center",
    ncol=len(labels),
    fontsize=TICK_SIZE,
    title_fontsize=TICK_SIZE
)

# Add spacing and global title
fig.subplots_adjust(top=0.85, bottom=0.28)

#fig.suptitle("Accuracy and ARI – Censoring Comparison", fontsize=TITLE_SIZE)

plt.savefig("Plots/Fig_Accuracy_ARI_All.pdf", bbox_inches="tight")
plt.close()
